# 🚀 BÁO CÁO: NHẬN DIỆN DEEPFAKE BẰNG UCF MODEL (ICCV 2023)
Notebook này bao gồm toàn bộ quy trình từ Tải Data, Tiền xử lý (Cắt ảnh), Huấn luyện và Đánh giá chuẩn Bài báo Khoa học.

## 1. Cài đặt Môi trường

In [ ]:
!pip install timm facenet-pytorch Pillow==9.5.0 -q
print("✅ Cài đặt môi trường thành công!")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, random, time, shutil
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from facenet_pytorch import MTCNN
from tqdm.notebook import tqdm
from sklearn.metrics import roc_curve, auc, accuracy_score
from scipy.optimize import brentq
from scipy.interpolate import interp1d

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device: {DEVICE} | Torch: {torch.__version__}")

## 2. Kết nối Dataset FaceForensics++ (C23)

In [ ]:
import os
import glob

# ── Paths giống V4 ──
DRIVE_ROOT   = '/content/drive/MyDrive/DoAn_Nhom4'
FF_ZIP       = f'{DRIVE_ROOT}/FaceForensics.zip'
EXTRACT_DIR  = '/content/dataset'

os.makedirs(EXTRACT_DIR, exist_ok=True)

# Giải nén FaceForensics++ từ Drive nếu chưa có
ff_folders = glob.glob(os.path.join(EXTRACT_DIR, 'FaceForensics*'))
if not ff_folders:
    print('Extracting FaceForensics++...')
    get_ipython().system('unzip -q "{FF_ZIP}" -d "{EXTRACT_DIR}/"')
    ff_folders = glob.glob(os.path.join(EXTRACT_DIR, 'FaceForensics*'))

# Tự động dò thư mục gốc
BASE = ff_folders[0]
if 'original' not in os.listdir(BASE):
    if 'FaceForensics++_C23' in os.listdir(BASE):
        BASE = os.path.join(BASE, 'FaceForensics++_C23')
    else:
        for root, dirs, _ in os.walk(BASE):
            if 'original' in dirs:
                BASE = root
                break

FAKE_DIRS = {
    'Deepfakes':      os.path.join(BASE, 'Deepfakes'),
    'Face2Face':      os.path.join(BASE, 'Face2Face'),
    'FaceSwap':       os.path.join(BASE, 'FaceSwap'),
    'NeuralTextures': os.path.join(BASE, 'NeuralTextures'),
}
REAL_DIR = os.path.join(BASE, 'original')
print(f"✅ Đã trỏ chính xác vào tập dữ liệu FF++ tại {BASE}!")

## 3. Tiền Xử Lý: Trích xuất khuôn mặt (MTCNN) + Backup Drive
Quá trình này tốn khoảng 15 phút. Hệ thống sẽ tự động khôi phục từ Drive nếu đã từng chạy qua để tiết kiệm thời gian!

In [ ]:
OUT_DIR = '/content/frames'
BACKUP_DIR = '/content/drive/MyDrive/UCF_frames_backup'
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(BACKUP_DIR, exist_ok=True)

MAX_VIDEOS_PER_CLASS = 1000   
FRAMES_PER_VIDEO     = 10     

folders = ['original', 'Deepfakes', 'Face2Face', 'FaceSwap', 'NeuralTextures']
mtcnn = MTCNN(margin=20, keep_all=False, post_process=False, device=DEVICE)

def extract_faces_mtcnn(video_list, save_path):
    os.makedirs(save_path, exist_ok=True)
    count = 0
    for vid_path in tqdm(video_list, desc=f"Cắt -> {os.path.basename(save_path)}"):
        cap = cv2.VideoCapture(vid_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames <= 0: cap.release(); continue
        target_frames = set([int(i * total_frames / FRAMES_PER_VIDEO) for i in range(FRAMES_PER_VIDEO)])
        vid_name = os.path.basename(vid_path).split('.')[0]
        cur = 0; saved = 0
        while True:
            ret = cap.grab()
            if not ret: break
            if cur in target_frames:
                ret, frame = cap.retrieve()
                if ret:
                    img_pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
                    face = mtcnn(img_pil)
                    if face is not None:
                        face_bgr = cv2.cvtColor(face.permute(1,2,0).cpu().numpy().astype(np.uint8), cv2.COLOR_RGB2BGR)
                        cv2.imwrite(os.path.join(save_path, f"{vid_name}_{saved}.jpg"), face_bgr)
                        count += 1
                saved += 1
                if saved >= FRAMES_PER_VIDEO: break
            cur += 1
        cap.release()
    return count

for folder in folders:
    src_folder = os.path.join(OUT_DIR, folder)
    bup_folder = os.path.join(BACKUP_DIR, folder)
    
    # 1. Nếu trên Colab chưa có, nhưng Drive có Backup -> Khôi phục
    if not os.path.exists(src_folder) and os.path.exists(bup_folder):
        print(f"♻️ Khôi phục '{folder}' từ Drive...")
        shutil.copytree(bup_folder, src_folder)
        
    # 2. Nếu Colab có rồi -> Bỏ qua
    elif os.path.exists(src_folder) and len(os.listdir(src_folder)) > 100:
        print(f"✅ Thư mục '{folder}' đã sẵn sàng.")
        
    # 3. Nếu cả 2 đều không có -> Bắt đầu cắt ảnh
    else:
        print(f"=== ĐANG CẮT ẢNH CHO '{folder}' ===")
        if folder == 'original':
            vids = [os.path.join(REAL_DIR, f) for f in os.listdir(REAL_DIR) if f.endswith('.mp4')]
        else:
            vids = [os.path.join(FAKE_DIRS[folder], f) for f in os.listdir(FAKE_DIRS[folder]) if f.endswith('.mp4')]
        random.shuffle(vids)
        extract_faces_mtcnn(vids[:MAX_VIDEOS_PER_CLASS], src_folder)
        
        # Cắt xong copy liền lên Drive để backup
        print(f"📦 Backup '{folder}' lên Drive...")
        shutil.copytree(src_folder, bup_folder, dirs_exist_ok=True)

print("🎉 HOÀN TẤT BƯỚC TIỀN XỬ LÝ ẢNH!")

## 4. Chuẩn bị DataLoader (Hyperparameters chuẩn bài báo)

In [ ]:
MANIP2IDX = {'Deepfakes': 0, 'Face2Face': 1, 'FaceSwap': 2, 'NeuralTextures': 3}

def collect_images(folder, label_binary, label_manip):
    paths = []
    if os.path.exists(folder):
        for f in os.listdir(folder):
            if f.lower().endswith('.jpg'):
                paths.append((os.path.join(folder, f), label_binary, label_manip))
    return paths

all_samples = []
all_samples += collect_images(os.path.join(OUT_DIR, 'original'), 0, 4)
for name in FAKE_DIRS.keys():
    all_samples += collect_images(os.path.join(OUT_DIR, name), 1, MANIP2IDX[name])

random.shuffle(all_samples)
n = len(all_samples)
assert n > 0, "❌ Không tìm thấy ảnh!"

n_train = int(0.8 * n); n_val = int(0.1 * n)
train_s = all_samples[:n_train]; val_s = all_samples[n_train:n_train+n_val]; test_s = all_samples[n_train+n_val:]
print(f"✅ Tổng ảnh: {n} | Train: {len(train_s)} | Val: {len(val_s)} | Test: {len(test_s)}")

# --- THÔNG SỐ HUẤN LUYỆN ---
IMG_SIZE   = 256    # Chuẩn kích thước UCF
BATCH_SIZE = 16     # Chống OOM cho GPU 16-22GB
EPOCHS     = 50     # Chuẩn hội tụ
LR         = 1e-4

train_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)), T.RandomHorizontalFlip(),
    T.ColorJitter(0.2, 0.2, 0.2, 0.05), T.RandomGrayscale(p=0.05),
    T.ToTensor(), T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
val_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

class FFDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples, self.transform = samples, transform
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, lb, lm = self.samples[idx]
        return self.transform(Image.open(path).convert('RGB')), lb, lm

train_loader = DataLoader(FFDataset(train_s, train_tf), BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(FFDataset(val_s,   val_tf),   BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(FFDataset(test_s,  val_tf),   BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

## 5. Kiến trúc UCF Model (Xception + Decoder)

In [ ]:
class XceptionEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        base = timm.create_model('xception', pretrained=True, num_classes=0, global_pool='')
        self.features = nn.Sequential(*list(base.children())[:-1])
        self.pool     = nn.AdaptiveAvgPool2d(1)
    def forward(self, x): return self.pool(self.features(x)).flatten(1)

class UCFModel(nn.Module):
    def __init__(self, feat_dim=512, num_manip=4):
        super().__init__()
        backbone_dim = 2048

        self.content_enc  = XceptionEncoder()
        self.content_proj = nn.Linear(backbone_dim, feat_dim)

        self.fp_enc        = XceptionEncoder()
        self.common_proj   = nn.Linear(backbone_dim, feat_dim)
        self.specific_proj = nn.Linear(backbone_dim, feat_dim)

        # Decoder 8x8 (Vá lỗi shape từ 224->256)
        self.decoder = nn.Sequential(
            nn.Linear(feat_dim * 2, 512 * 8 * 8), nn.Unflatten(1, (512, 8, 8)),
            nn.ConvTranspose2d(512, 256, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d(128,  64, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d( 64,  32, 4, 2, 1), nn.ReLU(),
            nn.ConvTranspose2d( 32,   3, 4, 2, 1), nn.Tanh(),
        )

        self.binary_cls = nn.Sequential(nn.Linear(feat_dim, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, 1))
        self.manip_cls  = nn.Sequential(nn.Linear(feat_dim, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, num_manip))

    def forward(self, x):
        content  = self.content_proj(self.content_enc(x))
        fp_feat  = self.fp_enc(x)
        common   = self.common_proj(fp_feat)
        specific = self.specific_proj(fp_feat)
        recon    = self.decoder(torch.cat([content, common], dim=1))
        bin_logit   = self.binary_cls(common).squeeze(1)
        manip_logit = self.manip_cls(specific)
        return bin_logit, manip_logit, recon, content, common, specific

model = UCFModel().to(DEVICE)
print(f"✅ Khởi tạo xong Model | Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

## 6. Training Pipeline (Anti-OOM)

In [ ]:
def contrastive_loss(common, specific, temperature=0.07):
    cn = F.normalize(common, dim=1); sn = F.normalize(specific, dim=1)
    sim = torch.sum(cn * sn, dim=1) / temperature
    return F.mse_loss(sim, torch.zeros_like(sim))

def reconstruction_loss(recon, target):
    mean = torch.tensor([0.485,0.456,0.406], device=target.device).view(1,3,1,1)
    std  = torch.tensor([0.229,0.224,0.225], device=target.device).view(1,3,1,1)
    t = (target * std + mean) * 2 - 1
    return F.l1_loss(recon, t)

bce_fn = nn.BCEWithLogitsLoss(); ce_fn  = nn.CrossEntropyLoss()
L_BIN = 1.0; L_MANI = 0.5; L_REC = 0.1; L_CON = 0.3

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

def train_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0; correct = 0; total = 0
    for imgs, lb, lm in loader:
        imgs = imgs.to(DEVICE); lb = lb.float().to(DEVICE); lm = lm.to(DEVICE)
        optimizer.zero_grad()
        bin_logit, manip_logit, recon, _, common, specific = model(imgs)
        l_bin  = bce_fn(bin_logit, lb)
        mask   = (lb == 1) & (lm < 4)
        l_mani = ce_fn(manip_logit[mask], lm[mask]) if mask.sum() > 0 else torch.tensor(0.).to(DEVICE)
        loss   = L_BIN*l_bin + L_MANI*l_mani + L_REC*reconstruction_loss(recon, imgs) + L_CON*contrastive_loss(common, specific)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        preds = (torch.sigmoid(bin_logit) > 0.5).long()
        correct += (preds == lb.long()).sum().item()
        total   += imgs.size(0)
    torch.cuda.empty_cache()  # Xả rác RAM cực kỳ quan trọng
    return total_loss / total, correct / total

def eval_epoch(model, loader):
    model.eval()
    total_loss = 0; all_probs = []; all_labels = []
    with torch.no_grad():
        for imgs, lb, lm in loader:
            imgs = imgs.to(DEVICE); lb = lb.float().to(DEVICE); lm = lm.to(DEVICE)
            bin_logit, manip_logit, recon, _, common, specific = model(imgs)
            mask   = (lb == 1) & (lm < 4)
            l_mani = ce_fn(manip_logit[mask], lm[mask]) if mask.sum() > 0 else torch.tensor(0.).to(DEVICE)
            loss   = L_BIN*bce_fn(bin_logit, lb) + L_MANI*l_mani + L_REC*reconstruction_loss(recon, imgs) + L_CON*contrastive_loss(common, specific)
            total_loss += loss.item() * imgs.size(0)
            all_probs.extend(torch.sigmoid(bin_logit).cpu().numpy())
            all_labels.extend(lb.cpu().numpy())
    probs = np.array(all_probs); labels = np.array(all_labels)
    fpr, tpr, _ = roc_curve(labels, probs)
    return total_loss / len(labels), accuracy_score(labels, (probs > 0.5).astype(int)), auc(fpr, tpr), probs, labels, fpr, tpr

## 7. Thực thi quá trình huấn luyện

In [ ]:
OUTPUT_DIR = '/content/drive/MyDrive'
os.makedirs(OUTPUT_DIR, exist_ok=True)
best_auc = 0
history  = {'tr_loss': [], 'vl_loss': [], 'tr_acc': [], 'vl_acc': [], 'vl_auc': []}

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc              = train_epoch(model, train_loader, optimizer)
    vl_loss, vl_acc, vl_auc, *_ = eval_epoch(model, val_loader)
    scheduler.step()
    
    for k, v in zip(history.keys(), [tr_loss, vl_loss, tr_acc, vl_acc, vl_auc]): history[k].append(v)
    
    if vl_auc > best_auc:
        best_auc = vl_auc
        torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, 'ucf_best.pth'))
        tag = " ← 🏆 BEST"
    else: tag = ""
    print(f"Epoch {epoch:02d} | Loss: {tr_loss:.4f}/{vl_loss:.4f} | Acc: {tr_acc:.4f}/{vl_acc:.4f} | AUC: {vl_auc:.4f} | {time.time()-t0:.0f}s{tag}")

## 8. Đánh giá chất lượng (Academic Quality Plots)

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 14, 'font.family': 'sans-serif', 'figure.dpi': 300, 'savefig.dpi': 300, 'axes.linewidth': 1.5, 'lines.linewidth': 2.5})
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
sns.set_palette("husl")

axes[0].plot(history['tr_loss'], label='Train Loss', alpha=0.9); axes[0].plot(history['vl_loss'], label='Val Loss', alpha=0.9)
axes[0].set_title('Loss Evolution', fontweight='bold'); axes[0].legend(frameon=True, shadow=True)

axes[1].plot(history['tr_acc'], label='Train Acc', alpha=0.9); axes[1].plot(history['vl_acc'], label='Val Acc', alpha=0.9)
axes[1].set_title('Accuracy Evolution', fontweight='bold'); axes[1].legend(frameon=True, shadow=True)

axes[2].plot(history['vl_auc'], label='Val AUC', color='#2ca02c', alpha=0.9)
axes[2].set_title('AUC Evolution', fontweight='bold'); axes[2].legend(frameon=True, shadow=True)

plt.tight_layout(pad=2.0)
plt.savefig(os.path.join(OUTPUT_DIR, 'ucf_training_curves.pdf'), format='pdf', bbox_inches='tight')
plt.show()

In [ ]:
model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, 'ucf_best.pth'), map_location=DEVICE))
_, test_acc, test_auc, probs, labels, fpr, tpr = eval_epoch(model, test_loader)
eer = brentq(lambda x: 1. - x - interp1d(fpr, tpr)(x), 0., 1.)

print(f"==================================================")
print(f"📊 IN-DATASET EVALUATION (FaceForensics++ C23)")
print(f"🎯 Accuracy : {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"🎯 AUC      : {test_auc:.4f}")
print(f"🎯 EER      : {eer:.4f}")
print(f"==================================================")

plt.figure(figsize=(8, 8))
plt.plot(fpr, tpr, color='#d62728', lw=3, label=f'UCF (AUC = {test_auc:.4f} | EER = {eer:.4f})')
plt.plot([0,1],[0,1], color='navy', lw=2, linestyle='--', alpha=0.6)
plt.scatter([eer], [1-eer], s=200, marker='*', color='gold', edgecolor='black', zorder=5, label='Equal Error Rate')
plt.xlim([-0.02, 1.02]); plt.ylim([-0.02, 1.02])
plt.xlabel('False Positive Rate', fontweight='bold'); plt.ylabel('True Positive Rate', fontweight='bold')
plt.title('Receiver Operating Characteristic (ROC)\nFaceForensics++ C23', fontweight='bold', pad=15)
plt.legend(loc='lower right', frameon=True, shadow=True, fontsize=12)
plt.savefig(os.path.join(OUTPUT_DIR, 'ucf_roc_ffpp.pdf'), format='pdf', bbox_inches='tight')
plt.show()

## 9. Cross-Dataset Evaluation (Celeb-DF-v2)

In [ ]:
# Hướng dẫn test chéo trên Celeb-DF (Đúng tiêu chuẩn bài báo UCF)
"""
# BƯỚC 1: Tải bộ dữ liệu Celeb-DF-v2 và cắt ảnh tương tự như FF++
# BƯỚC 2: Khởi tạo DataLoader

CELEB_DF_DIR = '/content/drive/MyDrive/CelebDF_frames'
celeb_samples = []
celeb_samples += collect_images(os.path.join(CELEB_DF_DIR, 'real'), label_binary=0, label_manip=4)
celeb_samples += collect_images(os.path.join(CELEB_DF_DIR, 'fake'), label_binary=1, label_manip=0) 
celeb_loader = DataLoader(FFDataset(celeb_samples, val_tf), BATCH_SIZE, shuffle=False, num_workers=2)

# BƯỚC 3: Đánh giá bằng UCF Model đã train
_, celeb_acc, celeb_auc, probs, labels, fpr, tpr = eval_epoch(model, celeb_loader)
celeb_eer = brentq(lambda x: 1. - x - interp1d(fpr, tpr)(x), 0., 1.)

print(f"📊 CROSS-DATASET EVALUATION (Celeb-DF v2)")
print(f"🎯 Accuracy : {celeb_acc:.4f} | AUC: {celeb_auc:.4f} | EER: {celeb_eer:.4f}")
"""
print("✅ Cấu trúc Test chéo (Cross-Dataset) đã sẵn sàng. Hãy bỏ comment khi bạn có ảnh Celeb-DF!")